In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import os
import sys
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

In [50]:
import torch
import tiktoken
from utility import token_ids_to_text, text_to_token_ids
from model.architecture import generate_text_simple, GPTModelFlashAttn
from model.dummy_model import load_yaml

start_context = "Every effort moves you"
tokenizer = tiktoken.get_encoding("gpt2")
config = load_yaml("../model/config/model_config_train.yaml")["model"]
model = GPTModelFlashAttn(config)

token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(start_context, tokenizer),
    max_new_tokens=10,
    context_size=config["context_length"]
)
print(f"Output text: {token_ids_to_text(token_ids, tokenizer)}")

Output text: Every effort moves you unison393 commissioner chars False IndianapolisListen secretary Fraziercise


## Calculating text generation loss

In [51]:
input1 = "every effort moves"
input2 = "I really like"
enc1 = torch.tensor(tokenizer.encode(input1))
enc2 = torch.tensor(tokenizer.encode(input2))
inputs = torch.stack((enc1, enc2))
print(inputs)
print(inputs.shape)

target1 = " effort moves you"
target2 = " really like chocolate"
t_enc1 = torch.tensor(tokenizer.encode(target1))
t_enc2 = torch.tensor(tokenizer.encode(target2))
targets = torch.stack((t_enc1, t_enc2))
print(targets)
print(targets.shape)

tensor([[16833,  3626,  6100],
        [   40,  1107,   588]])
torch.Size([2, 3])
tensor([[ 3626,  6100,   345],
        [ 1107,   588, 11311]])
torch.Size([2, 3])


In [52]:
with torch.no_grad():
    logits = model(inputs)

probas = torch.softmax(logits, dim=-1)
print(probas.shape)


torch.Size([2, 3, 50257])


In [53]:
token_ids = torch.argmax(probas, dim=-1, keepdim=True)
print(token_ids)
print(token_ids.shape)

tensor([[[46030],
         [  585],
         [45820]],

        [[15821],
         [48033],
         [ 9175]]])
torch.Size([2, 3, 1])


In [54]:
print(f"Targets batch 1: {token_ids_to_text(targets[0], tokenizer)}")
print(f"Outputs batch 1: {token_ids_to_text(token_ids[0].flatten(), tokenizer)}")

Targets batch 1:  effort moves you
Outputs batch 1: loadsordMAG


In [55]:
# print the initial softmax scores of the 2 sets of 3 tokens
text_idx = 0
target_probas1 = probas[text_idx, [0, 1, 2], targets[text_idx]]
print(f"Text 1 scores: {target_probas1}")

text_idx = 1
target_probas2 = probas[text_idx, [0, 1, 2], targets[text_idx]]
print(f"Text 2 scores: {target_probas2}")

Text 1 scores: tensor([1.0617e-05, 7.3194e-06, 1.8693e-05])
Text 2 scores: tensor([1.9585e-05, 2.9313e-05, 3.5361e-05])


In [56]:
# to get probability scores, 6 steps:
# logits -> probabilities -> target probabilities -> log probabilities -> avg log probability -> negative avg probabilit

# step 4:
log_probas = torch.log(torch.cat((target_probas1, target_probas2)))
print(log_probas)

tensor([-11.4530, -11.8250, -10.8873, -10.8408, -10.4375, -10.2499])


In [57]:
# step 5: average log prob
avg_log_probas = torch.mean(log_probas)
print(avg_log_probas)

tensor(-10.9489)


In [58]:
# step 6: turn neg (cross entropy loss)
neg_avg_log_probas = avg_log_probas * -1
print(neg_avg_log_probas)

tensor(10.9489)


In [59]:
# for cross entropy loss we need to flatten the tokens along the batch
# this means:
# logits dim  [2, 3, 50257] ---> [6, 50257]
# targets dim [2, 3] ----------> [6]

logits_flat = logits.flatten(0, 1)
print(logits_flat.shape)
targets_flat = targets.flatten(0)
print(targets.shape)

torch.Size([6, 50257])
torch.Size([2, 3])


In [60]:
loss = torch.nn.functional.cross_entropy(logits_flat, targets_flat)
print(loss)

tensor(10.9489)


In [62]:
perplexity = torch.exp(loss)
print(perplexity)

tensor(56892.6914)


In [16]:
# the above perplexity means that the model is uncertain about which among number of <perplexity val> 
# tokens to generate

### Training and valuation loss across dataset

In [63]:
file_path = "../1984.txt"
with open(file_path,"r", encoding="utf-8") as file:
    text = file.read()

In [64]:
total_len = len(text)
print(total_len)
total_tokens = len(tokenizer.encode(text, allowed_special={"<|endoftext|>"}))
print(total_tokens)

587006
141185


In [65]:
# 90% for training and 10% for validation
train_ratio = 0.9
split_idx = int(train_ratio * len(text))
train_data = text[:split_idx]
val_data = text[split_idx:]

In [66]:
from dataloader.dataloader import Train_dataloader
from model.dummy_model import load_yaml
torch.manual_seed(123)
config = load_yaml("../model/config/model_config_train.yaml")["model"]

train_loader = Train_dataloader(
    config,
    train_data,
    batch_size=4,
    shuffle=True,
    drop_last=True,
    num_workers=0,
    tokenizer=tokenizer
)
train_loader = train_loader.get_dataloader()

val_loader = Train_dataloader(
    config,
    val_data,
    batch_size=4,
    shuffle=False,
    drop_last=False,
    num_workers=0,
    tokenizer=tokenizer
)
val_loader = val_loader.get_dataloader()

In [67]:
# print shapes to verify
print("train dataloader")
for x, y in train_loader:
    print(x.shape, y.shape)

print("val dataloader")
for x, y in val_loader:
    print(x.shape, y.shape)

train dataloader
torch.Size([4, 256]) torch.Size([4, 256])
torch.Size([4, 256]) torch.Size([4, 256])
torch.Size([4, 256]) torch.Size([4, 256])
torch.Size([4, 256]) torch.Size([4, 256])
torch.Size([4, 256]) torch.Size([4, 256])
torch.Size([4, 256]) torch.Size([4, 256])
torch.Size([4, 256]) torch.Size([4, 256])
torch.Size([4, 256]) torch.Size([4, 256])
torch.Size([4, 256]) torch.Size([4, 256])
torch.Size([4, 256]) torch.Size([4, 256])
torch.Size([4, 256]) torch.Size([4, 256])
torch.Size([4, 256]) torch.Size([4, 256])
torch.Size([4, 256]) torch.Size([4, 256])
torch.Size([4, 256]) torch.Size([4, 256])
torch.Size([4, 256]) torch.Size([4, 256])
torch.Size([4, 256]) torch.Size([4, 256])
torch.Size([4, 256]) torch.Size([4, 256])
torch.Size([4, 256]) torch.Size([4, 256])
torch.Size([4, 256]) torch.Size([4, 256])
torch.Size([4, 256]) torch.Size([4, 256])
torch.Size([4, 256]) torch.Size([4, 256])
torch.Size([4, 256]) torch.Size([4, 256])
torch.Size([4, 256]) torch.Size([4, 256])
torch.Size([4, 25

In [70]:
# utility function to calculate cross entropy loss on batches
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch = input_batch.to(device)
    target_batch = target_batch.to(device)
    logits = model(input_batch)
    loss = torch.nn.functional.cross_entropy(
        input=logits.flatten(0,1),
        target=target_batch.flatten()
    )
    return loss

In [73]:
len(train_loader)

124

In [ ]:
from trainer import Trainer
trainer = Trainer(model=model)
with torch.no_grad():
    train_loss = trainer.calc_loss_loader(dataloader=train_loader)
    val_loss = trainer.calc_loss_loader(dataloader=val_loader)

total loss: 10.997190475463867 / batch num: 0
total loss: 11.005585670471191 / batch num: 0
